In [1]:
import time, html, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import os

In [2]:
CSV_PATH = "../data/taxi_2019_04.csv"
PARQUET_PATH = "../data/taxi_2019_04.parquet"

metadata = pq.ParquetFile(PARQUET_PATH).metadata

In [4]:

# --- Métadonnées au niveau du fichier ---
print("File metadata :")
print(f"  Version du format Parquet : {metadata.format_version}")
print(f"  Créé par                  : {metadata.created_by}")
print(f"  Nombre de lignes          : {metadata.num_rows}")
print(f"  Nombre de row groups      : {metadata.num_row_groups}")
print(f"  Nombre de colonnes        : {metadata.num_columns}")
print(f"  Taille des métadonnées    : {metadata.serialized_size} octets")

# --- Métadonnées au niveau du schéma (incluant celles ajoutées par pandas) ---
print("\n Schema Métadonnées (arrow) :")
print(pq.read_schema(PARQUET_PATH))
if metadata.metadata:
    print("\n Metadata (key/Values) :")
    for key, value in metadata.metadata.items():
        print(f"  {key}: {value[:120]}{'...' if len(value) > 120 else ''}")

# --- Métadonnées au niveau des row groups et des colonnes : compression
# et encoding utilisés, concrètement, pour chaque colonne de chaque row
# group (Parquet peut choisir un encoding différent par colonne).
n_row_groups = 5
n_row_groups = min(n_row_groups, metadata.num_row_groups)
print("\nCompression and encoding by column and row group :")
for rg_index in range(n_row_groups):
    rg = metadata.row_group(rg_index)
    print(f"\n  Row group {rg_index} ({rg.num_rows} lines, "
          f"{rg.total_byte_size / 1024:.1f} Ko uncompressed) :")
    for col_index in range(rg.num_columns):
        col = rg.column(col_index)
        print(f"    {col.path_in_schema:<20} "
              f"compression={col.compression:<8} "
              f"encodings={col.encodings} "
              f"({col.total_compressed_size / 1024:.1f} Ko compressed / "
              f"{col.total_uncompressed_size / 1024:.1f} Ko row)")
 
 


File metadata :
  Version du format Parquet : 1.0
  Créé par                  : parquet-cpp version 1.5.1-SNAPSHOT
  Nombre de lignes          : 7433139
  Nombre de row groups      : 114
  Nombre de colonnes        : 18
  Taille des métadonnées    : 174492 octets

 Schema Métadonnées (arrow) :
vendor_id: string
pickup_at: timestamp[us]
dropoff_at: timestamp[us]
passenger_count: int8
trip_distance: float
rate_code_id: string
store_and_fwd_flag: string
pickup_location_id: int32
dropoff_location_id: int32
payment_type: string
fare_amount: float
extra: float
mta_tax: float
tip_amount: float
tolls_amount: float
improvement_surcharge: float
total_amount: float
congestion_surcharge: float
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2548

 Metadata (key/Values) :
  b'pandas': b'{"index_columns": [{"kind": "range", "name": null, "start": 0, "stop": 7433139, "step": 1}], "column_indexes": [{"name":'...
  b'ARROW:schema': b'/////6gOAAAQAAAAAA

In [5]:
## Size

In [6]:
csv_size = os.path.getsize(CSV_PATH)
parquet_size = os.path.getsize(PARQUET_PATH)
print("=== Disk size ===")
print(f"CSV     : {csv_size / 1024:.1f} Ko")
print(f"Parquet : {parquet_size / 1024:.1f} Ko")
print(f"Ratio   : {csv_size / parquet_size:.2f}x\n")
 
start = time.perf_counter()
df_csv = pd.read_csv(CSV_PATH)
csv_read_time = time.perf_counter() - start
 
start = time.perf_counter()
df_parquet = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
parquet_read_time = time.perf_counter() - start
 
print("=== Speed (all file) ===")
print(f"CSV     : {csv_read_time:.4f} s")
print(f"Parquet : {parquet_read_time:.4f} s\n")


=== Disk size ===
CSV     : 708726.5 Ko
Parquet : 124078.6 Ko
Ratio   : 5.71x

=== Speed (all file) ===
CSV     : 24.8943 s
Parquet : 3.6821 s



In [7]:
# COLUMN PRUNING

In [8]:
columns =  ['rate_code_id', 'tip_amount']

row_group = metadata.row_group(0)

start = time.perf_counter()
pd.read_parquet(PARQUET_PATH, engine="pyarrow")
full_read_time = time.perf_counter() - start
 
start = time.perf_counter()
pd.read_parquet(PARQUET_PATH, engine="pyarrow", columns=columns)
pruned_read_time = time.perf_counter() - start

start = time.perf_counter()
pd.read_csv(CSV_PATH)
csv_full_read_time = time.perf_counter() - start
 
start = time.perf_counter()
pd.read_csv(CSV_PATH, usecols=columns)
csv_pruned_read_time = time.perf_counter() - start

print(f"\nParquet, read all columns                                       : {full_read_time:.4f} s")
print(f"Parquet, read specific columns (Combined (mpg), Make, Model)    : {pruned_read_time:.4f} s")
print()
print(f"\n CSV, read all columns                                        : {csv_full_read_time:.4f} s")
print(f" CSV, read specific columns (Combined (mpg), Make, Model)     : {csv_pruned_read_time:.4f} s")



Parquet, read all columns                                       : 4.5129 s
Parquet, read specific columns (Combined (mpg), Make, Model)    : 0.8921 s


 CSV, read all columns                                        : 17.6824 s
 CSV, read specific columns (Combined (mpg), Make, Model)     : 7.2073 s


In [9]:
# PREDICATE PUSHDOWN

In [ ]:
FILTER_COLUMN = 'tip_amount'
FILTER_VALUE = 100

metadata = pq.ParquetFile(PARQUET_PATH).metadata

start = time.perf_counter()
df_full = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
df_filtered = df_full[df_full[FILTER_COLUMN] < FILTER_VALUE]
no_pushdown_time = time.perf_counter() - start
 
dataset = ds.dataset(PARQUET_PATH, format="parquet")
start = time.perf_counter()
table_filtered = dataset.to_table(filter=ds.field(FILTER_COLUMN) > FILTER_VALUE)
pushdown_time = time.perf_counter() - start
 
print(f"No pushdown : {no_pushdown_time:.4f} s")
print(f"With pushdown   : {pushdown_time:.4f} s")
print(f"Lignes result : {len(df_filtered)} (pandas) / {table_filtered.num_rows} (pushdown)")

start = time.perf_counter()
df_csv_full = pd.read_csv(CSV_PATH)
df_csv_filtered = df_csv_full[df_csv_full[FILTER_COLUMN] > FILTER_VALUE]
csv_filter_time = time.perf_counter() - start
 
print(f"\nStatistic per row group on '{FILTER_COLUMN}' :")
col_index = metadata.schema.names.index(FILTER_COLUMN)
for i in range(metadata.num_row_groups):
    stats = metadata.row_group(i).column(col_index).statistics
    if stats is not None:
        skip = stats.max < FILTER_VALUE
        print(f"  Row group {i}: min={stats.min}, max={stats.max} "
              f"-> {'ignoré' if skip else 'lu'}")

 
print(f"\nFiltre sur CSV : {csv_filter_time:.4f} s")
print(f"Lignes résultat : {len(df_csv_filtered)} (csv)")

